# MLP Cell by Cell Code (Fashion-MNIST, 4x512)

`MyCNN.py` -> `MyMLP.py`, `MyCNN` -> `MyMLP`  
Dataset: CIFAR-10 -> **Fashion-MNIST**  
Architecture: VGG-style conv stack -> **4 hidden layers x 512 nodes**

`MPGELU.py`, `PGELU.py`, `Trainer.py` are unchanged from the original CNN project.

## Cell 1: Install torchinfo

In [ ]:
!pip install torchinfo

## Cell 2: Import Libraries

In [ ]:
import os
import sys
from pathlib import Path
from functools import partial

#PyTorch
import torch as torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
import time
from torchinfo import summary

import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import random
from typing import Iterable, Callable

#sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

#Import torchvision
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torchinfo import summary

#Import tqdm for a cool progress bar
from tqdm.auto import tqdm

# Measure time
from timeit import default_timer as timer

seed = 143
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

print(f"PyTorch Version: {torch.__version__}")
print(f"Numpy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"SciPy Version: {scipy.__version__}")

if torch.cuda.is_available():
    device = "cuda"
    print(f"Device: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"Device: {torch.backends.mps.is_available()}")
else:
    device = "cpu"
    print(f"Device: {device}")


## Cell 3: Import Helper Functions

In [ ]:
mother_path = Path.cwd()
for folder in ["utility", "act_fns", "models"]:
    path = os.path.join(mother_path, folder)
    if path not in sys.path:
        sys.path.append(path)

from MPGELU import MPGELU
from PGELU import PGELU
from MyMLP import MyMLP              # was MyCNN
from DataTransforms import DataTransforms
from Trainer import Trainer
from LoadedModel import LoadedModel


## Cell 4: Activation Function Visualization (MP-GELU)

In [ ]:
x_np = np.linspace(-4.0, 4.0, 400)
x_tensor = torch.tensor(x_np, dtype=torch.float32)
s_values = [-100, 0.0, 100, ]
plt.figure(figsize=(9, 6))

for s_val in s_values:
    activation1 = MPGELU(s_param=s_val)
    activation2 = MPGELU(s_param=s_val, use_softplus=False)
    with torch.inference_mode():
        y_tensor1 = activation1(x_tensor)
        y_tensor2 = activation2(x_tensor)

    lam_val = (1.0 + F.softplus(torch.tensor(s_val, dtype=torch.float32))).item()
    plt.plot(x_np, y_tensor1.numpy(), label=f"s = {s_val}", linewidth=2.5)

plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.title("Modified Parametric GELU (MP-GELU) Activation Function", fontsize=15)
xticks = np.arange(-4, 5, 0.8)
plt.xticks(xticks, fontsize=12)
plt.xlabel("Input ($x$)", fontsize=12)
yticks = np.arange(-2, 5, 0.8)
plt.yticks(yticks, fontsize=12)
plt.ylabel("Output ($f(x)$)", fontsize=12)
plt.grid(True, linestyle=':', alpha=0.9)
plt.legend(fontsize=12)
plt.show()


## Cell 5: Data Processing (Fashion-MNIST)

Fashion-MNIST is bundled with torchvision, so no Drive mount / tar extraction is needed -- it downloads straight into `./data` on first run.

In [ ]:
transformer = DataTransforms(dataset='fashionmnist', use_augmentation=True, use_stats=False)
train_transform = transformer.get_train_transform()
test_transform = transformer.get_test_transform()

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=test_transform)

train_dataloader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=10, pin_memory=True, persistent_workers=True)
test_dataloader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=10, pin_memory=True, persistent_workers=True)


## Cell 6: Visualize Sample Images

In [ ]:
fig = plt.figure(figsize=(8, 8))
rows, columns = 4, 4

for i in range(1, rows * columns + 1):
    idx = torch.randint(0, len(train_data), size=[1]).item()
    img, label = train_data[idx][0], train_data[idx][1]
    fig.add_subplot(rows, columns, i)

    # Fashion-MNIST is single-channel: squeeze the channel dim and use cmap='gray'
    plt.imshow(img.squeeze(0), cmap='gray')
    plt.title(train_data.classes[label])
    plt.grid(False)
    plt.axis(False)

plt.axis(False)
plt.show()


## Cell 7: Set its Parameters (4 hidden layers x 512 nodes)

In [ ]:
chosen_model_params = [512, 512, 512, 512]   # 4 hidden layers, 512 nodes each

model = MyMLP(input_shape=28 * 28,     # 784, flattened Fashion-MNIST image
              output_shape=10,
              activation=MPGELU,
              params=chosen_model_params).to(device)

print(summary(model=model,
       input=torch.randn(size=(1, 28, 28)).unsqueeze(dim=0)))

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")


## Cell 8: Test Dummy Input

In [ ]:
dummy_input = torch.randn(1, 1, 28, 28).to(device)
model(dummy_input)


## Cell 9: Loss / Accuracy / Training Params

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

def calculate_accuracy(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    accuracy = (correct / len(y_pred)) * 100
    return accuracy

num_of_epochs = 100
num_loss_steps = 5


## Cell 10: Execution and Saving

In [ ]:
torch.manual_seed(143)
activation_dict = {
    "PGELU": PGELU,
    "MP_GELU": partial(MPGELU, s_param=0.0, use_softplus=True),
    "ReLU": torch.nn.ReLU,
    "GELU": torch.nn.GELU,
    "LeakyReLU": torch.nn.LeakyReLU
}

all_results = {}

for act_name, act_fn in activation_dict.items():
    print(f"\n=======================================================")
    print(f"       TRAINING WITH ACTIVATION: {act_name}")
    print(f"=======================================================")

    model = MyMLP(input_shape=28 * 28, output_shape=10, activation=act_fn, params=chosen_model_params).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    trainer = Trainer(model=model, loss_fn=loss_fn, optimizer=optimizer,
                      calculate_accuracy=calculate_accuracy, device=device, loss_steps=num_loss_steps)

    results = {
        "train_loss": [],
        "train_accu": [],
        "test_loss": [],
        "test_accu": [],
        "activation_params": {},
        "grad_norm": []
    }

    start_time = time.perf_counter()

    for epoch in tqdm(range(num_of_epochs), desc=f"Training {act_name}"):
        if epoch % num_loss_steps == 0:
            print(f"Epoch: {epoch} \n =====================================================================")
        train_loss, train_accu, norm = trainer.train(data_loader=train_dataloader, epoch=epoch)
        test_loss, test_accu = trainer.test(data_loader=test_dataloader, epoch=epoch)

        results["train_loss"].append(train_loss)
        results["train_accu"].append(train_accu)
        results["test_loss"].append(test_loss)
        results["test_accu"].append(test_accu)
        results["grad_norm"].append(norm)

        # Track dynamic activation parameters
        for name, param in model.named_parameters():
            if 'alpha_param' in name or 'beta_param' in name or 's_param' in name:
                if name not in results.setdefault("activation_params", {}):
                    results["activation_params"][name] = []
                results["activation_params"][name].append(param.item())

    total_time = time.perf_counter() - start_time
    results["total_runtime"] = total_time
    all_results[act_name] = results

    # Save Model Checkpoint
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'results': results,
        'total_runtime': total_time,
        'activation_name': act_name
    }

    save_dir = "model_trained_params_new"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"chkpt5_{act_name}.pth")

    torch.save(checkpoint, save_path)
    print(f"Saved model state to {save_path}")


## Cell 11: Loading the Trained Model Params

In [ ]:
activation_dict = {
    "MP_GELU": MPGELU,
    "PGELU": PGELU,
    "ReLU": torch.nn.ReLU,
    "GELU": torch.nn.GELU,
    "LeakyReLU": torch.nn.LeakyReLU
}

loader = LoadedModel(
    activation_dict=activation_dict,
    model_params=chosen_model_params,
    input_shape=28 * 28,     # MyMLP needs the flattened input size
    output_shape=10,
    initial_fileName="chkpt5",
    checkpoint_dir="model_trained_params_new",
    device=device
)

my_models = loader.load_checkpoints()


## Cell 12: Plotting Setup

In [ ]:
to_plot = {
    "MP_GELU":   {"results": my_models["MP_GELU"]["results"],   "linewidth": 3.0, "color": "#000000"},
    "PGELU":     {"results": my_models["PGELU"]["results"],     "linewidth": 2.0, "color": "#E62937"},
    "ReLU":      {"results": my_models["ReLU"]["results"],      "linewidth": 2.0, "color": "#009E73"},
    "GELU":      {"results": my_models["GELU"]["results"],      "linewidth": 2.0, "color": "#4363D8"},
    "LeakyReLU": {"results": my_models["LeakyReLU"]["results"], "linewidth": 2.0, "color": "#F58231"}
}


## Cell 13: Alpha and Beta of PGELU

Parameter name changed: `conv_layers.1.*` -> `fc_layers.1.*`, since in `MyMLP.build_layers` the first `Linear` is at index 0 and its activation sits at index 1 of the `fc_layers` Sequential -- same position logic as the CNN's `conv_layers.1`.

In [ ]:
alpha = my_models["PGELU"]["results"]["activation_params"]["fc_layers.1.alpha_param"]
beta = my_models["PGELU"]["results"]["activation_params"]["fc_layers.1.beta_param"]

fig, ax = plt.subplots(1, 2, figsize=(10, 6))

ax[0].plot(alpha, label="Alpha\nParameter", color="#E62937", linewidth=2.5)
ax[0].grid(True, linestyle=':', alpha=0.9)
ax[0].tick_params(axis='x', labelsize=13)
ax[0].legend(loc="center left", bbox_to_anchor=(0.02, 0.5), fontsize=15)

ax[1].plot(beta, label="Beta\nParameter", color="#009E73", linewidth=2.5)
ax[1].grid(True, linestyle=':', alpha=0.9)
ax[1].tick_params(axis='x', labelsize=13)
ax[1].legend(loc="center left", bbox_to_anchor=(0.02, 0.5), fontsize=15)

fig.supxlabel("Epochs", fontsize=15)
fig.supylabel("Parameter Value", fontsize=15)
fig.suptitle("Dynamic Activation Parameters Evolution for PGELU", fontsize=16)

plt.tight_layout()
plt.show()


## Cell 14: Lambda of MPGELU

Parameter name changed: `conv_layers.1.s_param` -> `fc_layers.1.s_param`

In [ ]:
params_dict = my_models["MP_GELU"]["results"]["activation_params"]

s_vals = params_dict["fc_layers.1.s_param"]

s_values = torch.tensor(s_vals, dtype=torch.float32, device=device)

lam = 1.0 + F.softplus(s_values)

plt.figure(figsize=(11, 7))
plt.plot(lam.detach().cpu().numpy(), linewidth=2.5, color="#000055", label=r"Learned $\lambda$")

plt.xlabel("Epoch", fontsize=15)
plt.ylabel(r"Learned $\lambda$", fontsize=15)
plt.title(r"Evolution of Learned $\lambda$ in MP-GELU During Training", fontsize=16)

plt.xlim(0, 100)
plt.xticks(np.arange(0, 101, 5), fontsize=12)
plt.yticks(fontsize=12)

plt.grid(True, linestyle=":", alpha=0.9)
plt.legend(fontsize=14)

plt.tight_layout()
plt.show()


## Cell 15: Training Loss Curves

In [ ]:
plt.figure(figsize=(11, 7))

for act_name, property in to_plot.items():
    plot_results = property["results"]

    arguments = {
        "linewidth": property.get("linewidth", 1.0),
        "linestyle": property.get("linestyle"),
        "label": act_name,
        "color": property["color"]
    }

    plt.plot(plot_results["train_loss"], **arguments)

plt.legend(loc="upper right", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

plt.xticks(np.arange(0, 101, 5), fontsize=12)
plt.yticks(fontsize=12)

plt.xlim(0, 100)

plt.title("Training Loss Comparison Across Activation Functions", fontsize=16)

plt.tight_layout()
plt.show()


## Cell 16: Test Loss Curves

In [ ]:
plt.figure(figsize=(11, 7))

for act_name, property in to_plot.items():
    plot_results = property["results"]

    arguments = {
        "linewidth": property.get("linewidth", 1.0),
        "linestyle": property.get("linestyle"),
        "label": act_name,
        "color": property["color"]
    }

    plt.plot(plot_results["test_loss"], **arguments)

plt.legend(loc="upper right", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

plt.xticks(np.arange(0, 101, 5), fontsize=12)
plt.yticks(fontsize=12)

plt.xlim(0, 100)

plt.title("Test Loss Comparison Across Activation Functions", fontsize=16)

plt.tight_layout()
plt.show()


## Cell 17: Training Accuracy

In [ ]:
plt.figure(figsize=(11, 7))

for act_name, property in to_plot.items():
    plot_results = property["results"]

    arguments = {
        "linewidth": property.get("linewidth", 1.0),
        "linestyle": property.get("linestyle"),
        "label": act_name,
        "color": property["color"]
    }

    plt.plot(plot_results["train_accu"], **arguments)

plt.legend(loc="lower right", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

plt.xticks(np.arange(0, 101, 5), fontsize=12)
plt.yticks(np.arange(0, 101, 10), fontsize=12)

plt.xlim(0, 100)
plt.ylim(0, 100)

plt.title("Training Accuracy Comparison Across Activation Functions", fontsize=16)

plt.tight_layout()
plt.show()


## Cell 18: Test Accuracy

In [ ]:
plt.figure(figsize=(11, 7))

for act_name, property in to_plot.items():
    plot_results = property["results"]

    arguments = {
        "linewidth": property.get("linewidth", 1.0),
        "linestyle": property.get("linestyle"),
        "label": act_name,
        "color": property["color"]
    }

    plt.plot(plot_results["test_accu"], **arguments)

plt.legend(loc="lower right", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

plt.xticks(np.arange(0, 101, 5), fontsize=12)
plt.yticks(np.arange(0, 101, 10), fontsize=12)

plt.xlim(0, 100)
plt.ylim(0, 100)

plt.title("Test Accuracy Comparison Across Activation Functions", fontsize=16)

plt.tight_layout()
plt.show()


## Cell 19: LeakyReLU Gradient Norm Quick-Plot

In [ ]:
plt.plot(my_models["LeakyReLU"]["results"]["grad_norm"])


## Cell 20: F1 Scores

In [ ]:
f1_results = {}

print("Calculating F1 Scores...")
print("========================")

for act_name, model_data in my_models.items():
    model = model_data["model"]
    model.eval()

    all_preds = []
    all_targets = []

    with torch.inference_mode():
        for inputs, targets in test_dataloader:
            inputs = inputs.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())

    f1 = f1_score(all_targets, all_preds, average='macro')
    f1_results[act_name] = f1

    print(f"{act_name: >10}: {f1:.4f}")


## Cell 21: Printing Gradient Norms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 80)
print("GRADIENT L2 NORMS BY EPOCH")
print("=" * 80)

for act_name, results in all_results.items():

    print(f"\n{'=' * 80}")
    print(f"ACTIVATION: {act_name}")
    print(f"{'=' * 80}")

    grad_norms = np.array(results["grad_norm"], dtype=float)

    for epoch, grad_norm in enumerate(grad_norms, start=1):
        print(
            f"Epoch {epoch:3d}: "
            f"Gradient L2 Norm = {grad_norm:.6f}"
        )

    min_grad = np.min(grad_norms)
    max_grad = np.max(grad_norms)
    mean_grad = np.mean(grad_norms)
    std_grad = np.std(grad_norms, ddof=1)

    min_epoch = np.argmin(grad_norms) + 1
    max_epoch = np.argmax(grad_norms) + 1

    print(f"\n{'-' * 80}")
    print(f"SUMMARY -- {act_name}")
    print(f"{'-' * 80}")

    print(f"Minimum Gradient L2 Norm : {min_grad:.6f} (Epoch {min_epoch})")
    print(f"Maximum Gradient L2 Norm : {max_grad:.6f} (Epoch {max_epoch})")
    print(f"Mean Gradient L2 Norm    : {mean_grad:.6f}")
    print(f"Standard Deviation       : {std_grad:.6f}")
    print(f"Mean +/- SD              : {mean_grad:.6f} +/- {std_grad:.6f}")


## Cell 22: Gradient vs Epoch

In [ ]:
plt.figure(figsize=(11, 7))

for act_name, properties in to_plot.items():

    plot_results = properties["results"]

    epochs = np.arange(1, len(plot_results["grad_norm"]) + 1)

    plt.plot(
        epochs,
        plot_results["grad_norm"],
        linewidth=properties["linewidth"],
        linestyle="-",
        label=act_name,
        color=properties["color"]
    )

plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Gradient L2 Norm", fontsize=13)

plt.title("Gradient L2 Norm vs Epoch", fontsize=16)

plt.legend(loc="upper right", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.6)

plt.xticks(np.arange(0, 101, 5), fontsize=12)

plt.yticks(fontsize=12)

plt.xlim(1, 100)

plt.tight_layout()
plt.show()
